# TP 1 — Installer son atelier data

**Durée : 2 h.** Vous connaissez SQL. Le travail consiste à préparer un environnement dans lequel vous pourrez collecter, traiter et livrer des données.

Suivre `TP1_ENVIRONNEMENT.md`. À la fin, le notebook doit fonctionner dans VS Code avec un noyau Colab. Les trois premières cellules sont fournies ; les trois requêtes finales sont à écrire.

Les données décrivent plusieurs réseaux électriques américains sur un semestre. Aucune connaissance préalable du secteur n'est nécessaire.

## 1. Vérifier où le code s'exécute
Exécuter cette cellule. Expliquer la différence entre le dossier Git sur votre ordinateur et le dossier affiché par le noyau.

In [ ]:
import sys
from pathlib import Path

print("Python :", sys.version.split()[0])
print("Dossier :", Path.cwd())
print("Dossier Colab présent :", Path("/content").exists())

## 2. Installer DuckDB et activer les cellules SQL
Exécuter les deux cellules fournies. Ne pas ajouter JupySQL ni `%load_ext sql`. La fonction fournie transmet simplement les cellules `%%sql` à DuckDB ; elle ne fait aucune analyse des données.

In [ ]:
%pip install -q "duckdb==1.4.3"

In [ ]:
from pathlib import Path

import duckdb
from IPython.core.magic import register_cell_magic
from IPython.display import display

# Chemins relatifs : les fichiers sont sur la machine qui exécute le noyau.
Path("data").mkdir(exist_ok=True)
Path("exports").mkdir(exist_ok=True)
ancienne_connexion = globals().get("db")
if ancienne_connexion is not None:
    ancienne_connexion.close()
db = duckdb.connect("data/atelier.duckdb")

# Petit utilitaire fourni : il permet d'écrire des cellules SQL.
@register_cell_magic
def sql(_line, cell):
    if cell.strip() and any(not x.strip().startswith("--") and x.strip() for x in cell.splitlines()):
        result = db.execute(cell)
        if result.description:
            display(result.fetch_df())

print("Dossier d'exécution :", Path.cwd())

In [ ]:
%%sql
SELECT 'Connexion SQL prête' AS message, 2 + 3 AS resultat;

## 3. Récupérer les données
Exécuter le téléchargement fourni, puis vérifier le nom et la taille du fichier. Le fichier fait environ 42 Mo. Ne pas le committer dans Git.

Le notebook conserve toutes les colonnes sous forme de texte pour commencer. Le typage fera partie du pipeline au TP 2.

In [ ]:
from urllib.request import urlretrieve

url = "https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_BALANCE_2024_Jan_Jun.csv"
fichier = Path("data/eia930_2024_s1.csv")

if not fichier.exists():
    temporaire = fichier.with_suffix(".part")
    urlretrieve(url, temporaire)
    temporaire.replace(fichier)

print("Fichier prêt :", fichier, "—", round(fichier.stat().st_size / 1_000_000, 1), "Mo")

In [ ]:
%%sql
CREATE OR REPLACE VIEW brut AS
SELECT *
FROM read_csv('data/eia930_2024_s1.csv', header = true, all_varchar = true);

SELECT * FROM brut LIMIT 5;

## Comprendre les colonnes

Avant de requêter, lire ce que les colonnes veulent dire. Les mesures sont des **puissances moyennes sur une heure**, en **mégawatts (MW)** : une heure à 1 000 MW correspond à 1 000 MWh d'énergie. Chaque ligne décrit une heure d'un **réseau** (« Balancing Authority »), c'est-à-dire l'organisme chargé d'équilibrer en temps réel la production et la consommation d'une zone du réseau électrique américain (par exemple CISO pour la Californie, ERCO pour le Texas).

**Identifier la ligne**

| Colonne | Signification fonctionnelle |
|---|---|
| `Balancing Authority` | Le réseau (code court). |
| `Data Date` | Le jour de la mesure, en heure locale du réseau, au format américain `mm/jj/aaaa`. |
| `Hour Number` | Numéro de l'heure dans la journée locale, de 1 à 24. |
| `Local Time at End of Hour` | **Fin** de l'heure mesurée, en heure locale du réseau : la ligne « 14:00 » décrit l'intervalle de 13 h à 14 h. |
| `UTC Time at End of Hour` | Le même instant en UTC. Il ne dépend pas du fuseau ni du changement d'heure : c'est lui qui permet de trier et de comparer des réseaux. |
| `Region` | Grande région géographique à laquelle le réseau est rattaché. |

**Consommation, production et échanges**

| Colonne | Signification fonctionnelle |
|---|---|
| `Demand (MW)` | La consommation à satisfaire sur la zone du réseau. |
| `Demand Forecast (MW)` | La prévision de cette consommation faite à l'avance par le réseau. |
| `Net Generation (MW)` | La production des centrales du réseau, **nette** de leur propre consommation interne. Elle peut parfois être négative. |
| `Total Interchange (MW)` | Le solde des échanges avec les réseaux voisins. Positif : le réseau **exporte** ; négatif : il **importe**. |
| `Sum(Valid DIBAs) (MW)` | Somme des échanges avec les réseaux directement interconnectés jugés valides. Contrôle technique, pas utilisé dans ces TP. |

L'équilibre à retenir : **demande ≈ production nette − échanges nets**. Un réseau qui importe a donc une demande supérieure à sa production.

**Production par filière** (`Net Generation (MW) from …`)

| Filière | Signification |
|---|---|
| `Coal` | Charbon. |
| `Natural Gas` | Gaz naturel. |
| `Nuclear` | Nucléaire. |
| `All Petroleum Products` | Pétrole et dérivés. |
| `Hydropower and Pumped Storage` | Hydraulique, y compris le stockage par pompage : la valeur peut être négative quand l'installation pompe de l'eau, donc consomme. |
| `Solar` | Solaire. |
| `Wind` | Éolien. |
| `Other Fuel Sources` | Autres sources, regroupées par l'EIA (par exemple biomasse ou géothermie). |
| `Unknown Fuel Sources` | Source non identifiée par le réseau. |

**Les variantes `(Imputed)` et `(Adjusted)`.** Pour beaucoup de colonnes, l'EIA publie aussi une version « imputée » (estimée quand la valeur déclarée manquait) et une version « ajustée » (corrigée après contrôle de qualité). Elles sont renseignées dans des proportions très différentes : les valeurs imputées sont rares, les valeurs ajustées sont présentes dans la grande majorité des lignes. Ces TP travaillent sur les valeurs **déclarées** par les réseaux, sans les mélanger avec ces variantes.

**Absence, zéro et négatif.** Une case vide signifie que la mesure n'a pas été renseignée : ce n'est pas un zéro. Un zéro est une vraie mesure. Une valeur négative n'est pas automatiquement une erreur (autoconsommation de la filière, pompage, échanges), mais elle doit être interprétée avant d'être utilisée.

Afficher la liste des 44 colonnes de la source avec leur type et repérer chacune dans les tableaux ci-dessus. Elles sont toutes en texte pour l'instant.

In [ ]:
%%sql
DESCRIBE brut;

## 4. Trois requêtes à écrire

**A. Inventaire.** Combien de lignes et de réseaux distincts contient le fichier ? Le réseau est identifié par `"Balancing Authority"`.

Utiliser `LIMIT` pour les aperçus : ne pas afficher toute la source dans le notebook.

In [ ]:
%%sql
-- À vous de coder : compter les lignes et les réseaux distincts de brut.

**B. Répartition.** Compter les lignes par réseau et trier le résultat. Tous les réseaux ont-ils le même nombre de lignes ? Ne rien supprimer à ce stade.

In [ ]:
%%sql
-- À vous de coder : compter les lignes par "Balancing Authority" et trier.

**C. Première sélection.** Choisir un réseau, afficher dix lignes avec date, fin d'heure locale, demande et production totale. Renommer les colonnes pour rendre le tableau lisible.

Colonnes : `"Data Date"`, `"Local Time at End of Hour"`, `"Demand (MW)"`, `"Net Generation (MW)"`.

In [ ]:
%%sql
-- À vous de coder : sélectionner dix lignes du réseau de votre choix avec des alias lisibles.

## Pour aller plus loin (si vous avez fini avant les autres)

Trois requêtes facultatives, toujours en SQL et sans graphique. Elles servent à mieux connaître la source avant le TP 2.

**D. Réseaux sans demande.** Lister les réseaux dont `"Demand (MW)"` est absente (valeur vide) sur au moins une ligne. Afficher pour chacun le nombre de lignes concernées et le pourcentage. Combien de réseaux ne déclarent jamais leur demande ? Quelle conséquence aurait ce constat sur une « demande moyenne nationale » ?

In [ ]:
%%sql
-- À vous de coder : réseaux avec une demande absente, effectif et pourcentage.

**E. Couverture temporelle.** Pour chaque réseau, afficher le nombre de lignes, le premier et le dernier horodatage UTC, et le nombre de décalages différents entre l'heure UTC et l'heure locale. Les horodatages sont du texte au format `%m/%d/%Y %I:%M:%S %p` : les convertir avec `TRY_STRPTIME`. Les réseaux n'ont pas tous le même nombre de lignes : à partir du résultat, formuler une explication, sans rien supprimer.

In [ ]:
%%sql
-- À vous de coder : couverture temporelle par réseau (lignes, premier/dernier UTC, nombre de décalages UTC/local).

**F. Demande et pointe.** Convertir `"Demand (MW)"` avec `TRY_CAST(... AS DOUBLE)`, puis afficher les cinq réseaux dont la demande moyenne est la plus élevée, avec leur pointe maximale. Pour le réseau que vous avez choisi en C, donner la date et l'heure locale de la pointe. Les valeurs absentes ne doivent pas être transformées en zéro.

In [ ]:
%%sql
-- À vous de coder : top 5 des réseaux par demande moyenne, puis pointe de votre réseau.

## Notes d'usage de l'IA (si utilisée)

| Question posée | Aide retenue | Vérification effectuée |
|---|---|---|
| … | … | … |

## 5. Expliquer et sauvegarder

À compléter ci-dessous :
- Le code s'exécute sur…
- Le notebook est sauvegardé dans…
- Les données téléchargées sont dans…
- Une ligne représente…
- Mon réseau choisi est…

Enregistrer le notebook, puis faire le commit demandé dans le support TP 1. Être capable d'expliquer une requête et de changer son filtre en direct.